# Load and process Transparent California Data

### Prerequisite

Download the data.zip folder from the google drive and extract it in the project directory.

The directory structure for this pipline should look like the following

```md
data/
    raw/
    processed/
data-pipline
    notebooks.ipynb
```

In [46]:
import os

# Check or create prerequisite folders
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/raw", exist_ok=True)
    


In [47]:
# Lossless data processing for provided downloaded from Transparent Califronia

import polars as pl

print("Scanning CSV files...")

# Numerical cols
financial_cols = [
    "base", "overtime", "other", "totalpay", "totalbenefits",
    "totalpaybenefits"
]

df = pl.scan_csv(
    "../data/raw/tc_raw/**/*.csv",
    include_file_paths="filepath"
)
# Empty Cols
df = df.drop(['ual', 'benefitsee', 'benefitser', 'benefitsdc'], strict=False)

# Helper function to currency text into Ints!
def clean_currency(col_name):
    return ( pl.col(col_name) .cast(pl.Int32, strict=False))

df = df.rename(str.lower)

# Apply typing and data transformations
df = df.with_columns(
    pl.selectors.string().str.to_lowercase()
).with_columns(
    # Get Year From Filename
    pl.col("filepath").str.extract(r"-(\d{4})\.csv$", 1).cast(pl.Int16).alias("year"),

    # Categorical cols
    pl.col("job").cast(pl.Categorical),
    pl.col("department").cast(pl.Categorical),

    # Clean and cast all financial columns
    *[clean_currency(col) for col in financial_cols]
)

df = df.drop(["filepath"])

df = df.drop_nulls(subset=["name"])

df.describe()

Scanning CSV files...


statistic,name,job,department,base,overtime,other,benefitshw,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status
str,str,str,str,f64,f64,f64,str,f64,f64,f64,str,f64,str,str
"""count""","""987543""","""987543""","""0""",987543.0,987543.0,987543.0,"""0""",987543.0,987543.0,987543.0,"""28458""",987543.0,"""987543""","""0"""
"""null_count""","""0""","""0""","""987543""",0.0,0.0,0.0,"""987543""",0.0,0.0,0.0,"""959085""",0.0,"""0""","""987543"""
"""mean""",null,null,null,38954.754078,251.729386,2990.511412,null,42197.157741,18286.612329,60484.049206,null,2019.001874,null,null
"""std""",null,null,null,41626.826402,2194.793662,7709.038972,null,44443.447239,20332.057299,63379.724403,null,3.35553,null,null
"""min""",""" judiyaba""",null,null,-5669.0,-72.0,-23732.0,null,0.0,-95066.0,-64657.0,"""employee held multiple positio…",2013.0,"""california state university""",null
"""25%""",null,null,null,412.0,0.0,0.0,null,2430.0,0.0,2500.0,null,2016.0,null,null
"""50%""",null,null,null,29019.0,0.0,2.0,null,32434.0,11413.0,46481.0,null,2019.0,null,null
"""75%""",null,null,null,63965.0,0.0,2500.0,null,67690.0,33840.0,100952.0,null,2022.0,null,null
"""max""","""zvonimir hlousek""",null,null,790125.0,249767.0,473046.0,null,970004.0,181837.0,1.114304e6,"""employee holds multiple positi…",2024.0,"""california state university""",null


In [48]:
import re

import pandas as pd

CSU_ROLES = [
    ("supplemental_teaching_pay", r"special programs|summer session|extension|additional employment|music studio"),
    ("ladder_faculty", r"^instructional faculty|^grant-related/specially funded instructional faculty|^department chair|^cma instructional faculty"),
    ("lecturer", r"lecturer"),
    ("clinical_adjunct_faculty", r"^visiting faculty"),
    ("postdoc", r"^research fellow|post-doctoral"),
    ("trainee", r"counselor intern"),
    ("researcher_specialist_librarian", r"librarian|academic-related|coach"),
]


def role_of(title, rules) -> str:
    t = str(title or "").lower().strip()
    return next((role for role, pattern in rules if re.search(pattern, t)), "staff")

df = df.collect().to_pandas()

df = df.copy()
df["role"] = df["job"].map(lambda t: role_of(t, CSU_ROLES)).astype("string")

# Replace "name" with your actual name-column name if needed
df["name"] = (
    df["name"]
    .str.title()
    .str.replace(r"(?<=\s)[A-Z]\.?(?=\s|$)", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# The way we are droping duplicates should be evaluated

# num_dups = df.duplicated(subset=["year", "name", "job"]).sum()
# print(f"Number of duplicate rows: {num_dups}")

# dups_df = df[df.duplicated(subset=["year", "name", "job"], keep=False)]

# print(dups_df)

# counts = df.value_counts(subset=["year", "name", "job"])
# print(counts[counts > 1])


# Remove Duplicate Values
df = df.drop_duplicates(subset=["year", "name", "job"])

df = pl.from_pandas(df)

df.describe()

statistic,name,job,department,base,overtime,other,benefitshw,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status,role
str,str,str,f64,f64,f64,f64,str,f64,f64,f64,str,f64,str,str,str
"""count""","""943556""","""943556""",0.0,943556.0,943556.0,943556.0,"""0""",943556.0,943556.0,943556.0,"""28344""",943556.0,"""943556""","""0""","""943556"""
"""null_count""","""0""","""0""",943556.0,0.0,0.0,0.0,"""943556""",0.0,0.0,0.0,"""915212""",0.0,"""0""","""943556""","""0"""
"""mean""",null,null,null,40398.748916,262.691425,3082.202439,null,43743.810833,18964.355998,62708.455843,null,2018.927074,null,null,null
"""std""",null,null,null,41752.510475,2242.975016,7817.771994,null,44541.706162,20405.017431,63506.469725,null,3.368411,null,null,null
"""min""","""A Alisia Nantze""",null,null,-5669.0,-72.0,-23732.0,null,0.0,-95066.0,-64657.0,"""employee held multiple positio…",2013.0,"""california state university""",null,"""clinical_adjunct_faculty"""
"""25%""",null,null,null,2310.0,0.0,0.0,null,3800.0,0.0,3960.0,null,2016.0,null,null,null
"""50%""",null,null,null,31876.0,0.0,54.0,null,35102.0,14047.0,51391.0,null,2019.0,null,null,null
"""75%""",null,null,null,65517.0,0.0,2694.0,null,69390.0,34498.0,103235.0,null,2022.0,null,null,null
"""max""","""Zvonimir Hlousek""",null,null,790125.0,249767.0,473046.0,null,970004.0,181837.0,1.114304e6,"""employee holds multiple positi…",2024.0,"""california state university""",null,"""trainee"""


# Attach Transparent California Data to Scrapped CSU Data

Make sure to import "all_csu_scraped_data_combined.parquet" from Raw data files and "transparent_california_no_dup_filtered_csu_salaries.parquet" from /Faculty Data/CSU Data/ transparent_california_no_dup_filtered_csu_salaries.parquet" into the Colab before you start


In [49]:
import pandas as pd

def prepare_names(df):
    df = df.copy()
    # Normalize both directory "Last, First" and payroll "First Last" names.
    # Strip suffixes before reordering so "Smith, John, Jr." stays well formed.
    names = (
        df["name"].astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace(r"(?i),?\s+(?:Jr\.?|Sr\.?|II|III|IV)$", "", regex=True)
        .str.replace(r"^([^,]+),\s*([^,]+)$", r"\2 \1", regex=True)
        # Remove middle initials only after converting to First Last order.
        # Keep full middle names, first initials, and compound surnames.
        .str.replace(r"(?<=\S) (?:(?:[A-Za-z]\.?|(?:[A-Za-z]\.){2,}) )+(?=\S)", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    df["name"] = names
    tokens = names.str.casefold().str.split()

    df["_match_full"] = tokens.str.join(" ")
    df["_match_first"] = tokens.str[0]
    df["_match_last"] = tokens.str[-1]
    df["_match_count"] = tokens.str.len()

    # Exclude missing or empty names so they cannot match each other.
    return df.loc[df["_match_count"].gt(0)].copy()

csu_pay = df.to_pandas()
raw_csu = pd.read_parquet('../data/raw/all_csu_scraped_data_combined.parquet')

csu_pay = csu_pay[csu_pay["role"] != "staff"]

pay = prepare_names(csu_pay)
faculty = prepare_names(raw_csu)


# Find candidates with matching first and last tokens.
candidates = pay.merge(
    faculty,
    on=["_match_first", "_match_last"],
    how="inner",
    suffixes=("_transparentCA", "_scraped"),
)

# If token counts are equal, the full names must also match.
keep = (
    candidates["_match_full_transparentCA"].eq(candidates["_match_full_scraped"])
    | candidates["_match_count_transparentCA"].ne(candidates["_match_count_scraped"])
)

joined = (
    candidates.loc[keep]
    .drop(columns=[
        "_match_first",
        "_match_last",
        "_match_full_transparentCA",
        "_match_full_scraped",
        "_match_count_transparentCA",
        "_match_count_scraped",
    ])
    .reset_index(drop=True)
)

joined = joined.drop(columns=["name_transparentCA"]).rename(
    columns={"name_scraped": "name"}
)

joined = joined[["name"] + [col for col in joined.columns if col != "name"]]

print(f"Matched rows: {len(joined):,}")
joined.head(10)


cleaned_columns = [
    "name", "job", "role", "base", "overtime", "other", "totalpay",
    "totalbenefits", "totalpaybenefits", "year_transparentCA",
  "title", "department_scraped", "school", "year_scraped",
    "major", "start_date",
]
cleaned_joined = joined.loc[:, cleaned_columns].copy()
print(f"Saved {len(cleaned_joined):,} cleaned joined rows")



Matched rows: 167,220
Saved 167,220 cleaned joined rows


<!-- DELETE THIS -->
### Load Data

In [50]:
import polars as pl
import duckdb

df = pl.from_pandas(cleaned_joined)

<!-- DELETE THIS -->
### Basic Data Information


<!-- DELETE THIS -->
### Role Column
No rows are dropped here. `role` comes from the Transparent California job title.
`filtered_faculty` is the stable input to department bucketing; `faculty_salaries` is the working table.

#### I am unsure about this sql statement

In [51]:
# Rebuild from the original input so this cell is safe to rerun.
df = duckdb.sql("SELECT DISTINCT * FROM df").pl()
duckdb.sql("CREATE OR REPLACE TABLE filtered_faculty AS SELECT * FROM df")
duckdb.sql("CREATE OR REPLACE TABLE faculty_salaries AS SELECT * FROM filtered_faculty")

<!-- DELETE THIS -->
Data info after adding roles

# Depatment Buckets

## Proposed Department Buckets
* Engineering
* Computer Science
* Science & Mathematics
* Medicine & Heath Sciences
* Business, Economics & Finance
* Agriculture, Food & Animal
* Architecture & Environmental Design
* Humanities & Languages
* Social Sciences & Communication
* Education & Public Administration

### Creating the Proposed Buckets
Rules use normalized department text and first-match precedence. Education rules precede the
humanities/social rules; `art`/`arts` must be complete words, and there is no generic `studies` rule.
Known subjects are mapped below. Missing or ambiguous departments stay in `Other` for review.
Each run rebuilds from `filtered_faculty`, avoiding stale or duplicated bucket columns.

In [52]:
bucket_sql = r"""
SELECT *, lower(trim(department_scraped)) as department,
CASE
    WHEN
        department ILIKE '%computer%'
        OR department ILIKE '%comp sci%'
        OR department ILIKE '%cmptr sci%'
        OR department ILIKE '%computing%'
        OR department ILIKE '%software%'
        OR department ILIKE '%informatics%'
        OR department ILIKE '%cyber%'
        OR department ILIKE '%data science%'
        OR department ILIKE '%information studies%'
        OR department ILIKE '%school of info%'
        OR department ILIKE '%information systems%'
        OR department ILIKE '%information technology%'
        OR department ILIKE '%decision and information sciences%'
    THEN 'Computer Science'

    WHEN
        department ILIKE '%medicine%'
        OR department ILIKE '%surgery%'
        OR department ILIKE '%vm:%'
        OR department ILIKE '%med:%'
        OR department ILIKE '%nurs%'
        OR department ILIKE '%dentistry%'
        OR department ILIKE '%pharma%'
        OR department ILIKE '%gynecology%'
        OR department ILIKE '%urology%'
        OR department ILIKE '%oncology%'
        OR department ILIKE '%family med%'
        OR department ILIKE '%radiation%'
        OR department ILIKE '%ophthalmology%'
        OR department ILIKE '%pathology%'
        OR department ILIKE '%peds%'
        OR department ILIKE '%dermatology%'
        OR department ILIKE '%pediatrics%'
        OR department ILIKE '%anesthesiology%'
        OR department ILIKE '%optometry%'
        OR department ILIKE '%psychiatry%'
        OR department ILIKE '%emerg med%'
        OR department ILIKE '%radiolog%'
        OR department ILIKE '%radiation%'
        OR department ILIKE '%anesthes%'
        OR department ILIKE '%public health%'
        OR department ILIKE '%med ed administration%'
        OR department ILIKE '%deans office-school of medicin%'
        OR department ILIKE '%semel institute%'
        OR department ILIKE '%school of med%'
        OR department ILIKE '%dom cardiology%'
        OR department ILIKE '%ophth clinical%'
        OR department ILIKE '%med-campus%'
        OR department ILIKE '%dntl sci%'
        OR department ILIKE '%ob/gyn%'
        OR department ILIKE '%ambulatory%'
        OR department ILIKE '%electronic health record%'
        OR department ILIKE '%electrocardiology%'
        OR department ILIKE '%epidemiology%'
        OR department ILIKE '%otolaryngology%'
        OR department ILIKE '%hematology%'
        OR department ILIKE '%school of public health%'
        OR department ILIKE '%health policy and management%'
        OR department ILIKE '%health sciences%'
        OR department ILIKE '%lppi: instr & research%'
        OR department ILIKE 'dom %'
        OR department ILIKE 'som %'
        OR department ILIKE 'som-%'
        OR department ILIKE '%obgyn%'
        OR department ILIKE '%sfgh%'
        OR department ILIKE 'psych-%'
        OR department ILIKE 'fcm-%'
        OR department ILIKE '%health science%'
        OR department ILIKE '%health care%'
        OR department ILIKE '%health serv%'
        OR department ILIKE '%pub health%'
        OR department ILIKE '%applied health%'
        OR department ILIKE '%kinesiology%'
        OR department ILIKE '%occupational therapy%'
        OR department ILIKE '%audiology%'
        OR department ILIKE '%clinical laboratory%'
        OR department ILIKE '%clinical science%'
        OR department ILIKE '%gerontology%'
        OR department ILIKE '%marital and family therapy%'
    THEN 'Medicine & Health Sciences'

    WHEN
        department ILIKE '%engineer%'
        OR department ILIKE '%engr%'
        OR department ILIKE '%mech%'
        OR department ILIKE '%civil%'
        OR department ILIKE '%aero%'
        OR department ILIKE '%electric%'
        OR department ILIKE '%soe-ams%'
        OR department ILIKE '%chemical%'
        OR department ILIKE '%industrial%'
        OR department ILIKE '%material%'
        OR department ILIKE '%soece-ee-tm-cm%'
        OR department ILIKE '%construction management%'
        OR department ILIKE '%concrete industry%'
    THEN 'Engineering'

    WHEN
        department ILIKE '%plant science%'
        OR department ILIKE '%agricultur%'
        OR department ILIKE '%espm%'
        OR department ILIKE '%nutrition%'
        OR department ILIKE '%inst of environment and sustai%'
        OR department ILIKE '%energy & resources group%'
        OR department ILIKE '%animal%'
        OR department ILIKE '%food science%'
        OR department ILIKE '%food safety%'
        OR department ILIKE '%family and consumer science%'
        OR department ILIKE '%family & consumer science%'
        OR department ILIKE '%forest%'
        OR department ILIKE '%wildlife%'
        OR department = 'agr academic instruction'
    THEN 'Agriculture, Food & Animal'

    WHEN
        department ILIKE '%architec%'
        OR department ILIKE '%planning%'
        OR department ILIKE '%urban%'
        OR department ILIKE '%landscape arch%'
        OR department ILIKE '%environmental%'
        OR department ILIKE '%school of the environment%'
        OR department ILIKE '%env sci%'
        OR department ILIKE '%env studies%'
        OR department ILIKE '%environ s%'
        OR department ILIKE '%interior design%'
    THEN 'Architecture & Environmental Design'

    WHEN
        department ILIKE '%business%'
        OR department ILIKE '%econ%'
        OR department ILIKE '%finance%'
        OR department ILIKE '%management%'
        OR department ILIKE '%marketing%'
        OR department ILIKE '%accounting%'
        OR department ILIKE '%accountancy%'
        OR department ILIKE '%financial%'
        OR department ILIKE '%entrepreneur%'
        OR department ILIKE '%hospitality%'
        OR department ILIKE '%tourism%'
        OR department ILIKE '%decision sciences%'
        OR department ILIKE '%real estate%'
        OR department ILIKE '%technology and operations%'
        OR department ILIKE '%supply chain%'
        OR department ILIKE '%organizational behavior%'
        OR department ILIKE '%organizational leadership%'
        OR department ILIKE '%global innovation and leadership%'
        OR department ILIKE '%recr, hosp%'
        OR department = 'bus'
        OR department ILIKE '%haas core%'
        OR department ILIKE '%technology management program%'
        OR department ILIKE '%grad.sch. int.relat./pac.stud.%'
    THEN 'Business, Economics & Finance'

    WHEN
        department ILIKE '%math%'
        OR regexp_matches(department, '\bstat(istics|istical)?\b')
        OR department ILIKE '%statistics%'
        OR department ILIKE '%bio%'
        OR department ILIKE '%neuro%'
        OR department ILIKE '%phys%'
        OR department ILIKE '%land air & water resources%'
        OR department ILIKE '%climate/atmos sci/phy oceanog%'
        OR department ILIKE '%ocean sciences%'
        OR department ILIKE '%entomology%'
        OR department ILIKE '%physics%'
        OR department ILIKE '%astro%'
        OR department ILIKE '%chem%'
        OR department ILIKE '%sio dep%'
        OR department ILIKE '%molecular, cell%'
        OR department ILIKE '%earth%'
        OR department ILIKE '%oceanic%'
        OR (department ILIKE '%ecology%' AND department NOT ILIKE '%human ecology%')
        OR department ILIKE '%genetics%'
        OR department ILIKE '%igpp%'
        OR department ILIKE '%geosciences%'
        OR department ILIKE '%oceanography%'
        OR department ILIKE '%geology%'
        OR department ILIKE '%viticulture%'
        OR department ILIKE '%nematology%'
        OR department ILIKE '%demography%'
        OR department ILIKE '%geological%'
        OR department ILIKE '%marine science%'
        OR department ILIKE '%marine facilities%'
        OR department ILIKE '%natural sciences%'
        OR department = 'college of sciences'
    THEN 'Science & Mathematics'

    WHEN
        department ILIKE '%education%'
        OR department ILIKE '%public policy%'
        OR department ILIKE '%policy&strategy%'
        OR department ILIKE 'jd program'
        OR department ILIKE '%law%'
        OR department ILIKE '%legal%'
        OR department ILIKE '%journalism%'
        OR department ILIKE '%teaching%'
        OR department ILIKE '%public administration%'
        OR department ILIKE '%public affairs%'
        OR department ILIKE '%pub pol%'
        OR department ILIKE '%homeland security%'
        OR department ILIKE '%student affairs%'
        OR department ILIKE '%learning design%'
        OR department ILIKE '%edleadership%'
        OR department ILIKE '%ed doctorate%'
        OR department ILIKE '%undergrad ed%'
        OR department ILIKE '%extended ed%'
        OR department ILIKE '%instructional technology%'
        OR department ILIKE '%lifelong learning%'
    THEN 'Education & Public Administration'

    WHEN
        department ILIKE '%language%'
        OR department ILIKE '%linguistics%'
        OR department ILIKE '%humanities%'
        OR department ILIKE '%english%'
        OR department ILIKE '%literature%'
        OR department ILIKE '%writing%'
        OR department ILIKE '%history%'
        OR department ILIKE '%philosophy%'
        OR department ILIKE '%french%'
        OR department ILIKE '%spanish%'
        OR department ILIKE '%german%'
        OR department ILIKE '%japanese%'
        OR department ILIKE '%dance%'
        OR department ILIKE '%african%'
        OR department ILIKE '%music%'
        OR department ILIKE '%division of arts%'
        OR department ILIKE '%classics%'
        OR department ILIKE '%film%'
        OR department ILIKE '%visual arts%'
        OR department ILIKE '%theatre%'
        OR department ILIKE '%religious%'
        OR department ILIKE '%theater%'
        OR department ILIKE '%asian lang%'
        OR department ILIKE '%eastern lang%'
        OR department ILIKE '%world arts%'
        OR department ILIKE '%southeast asian%'
        OR department ILIKE '%drama department%'
        OR department ILIKE '%department of chicana%'
        OR department ILIKE '%asian-american%'
        OR department ILIKE '%asian%'
        OR department ILIKE '%studio art%'
        OR regexp_matches(department, '\barts?\b')
        OR department ILIKE '%design program%'
        OR department ILIKE '%rhetoric%'
        OR department ILIKE '%near eastern%'
        OR department ILIKE '%chicano%'
        OR department ILIKE '%department of design%'
        OR department ILIKE '%media%'
        OR department ILIKE '%institute for society and gene%'
        OR department ILIKE '%college of creative studies%'
        OR department ILIKE '%american studies%'
        OR department ILIKE '%italian studies%'
        OR department ILIKE '%native american studies%'
        OR department ILIKE '%performance, play & design%'
        OR department ILIKE '%science & technology studies%'
        OR department ILIKE '%drama%'
        OR department ILIKE '%hispanic%'
        OR department ILIKE '%italian%'
        OR department ILIKE '%scandinavian%'
        OR department ILIKE '%slavic%'
        OR department ILIKE '%greek & roman%'
        OR department ILIKE '%cinema%'
        OR department ILIKE '%religion%'
        OR department ILIKE '%photography%'
        OR department ILIKE '%liberal studies%'
        OR department ILIKE '%lib studies%'
        OR department ILIKE '%arabic%'
        OR department ILIKE '%chinese%'
        OR department ILIKE '%russian%'
        OR department ILIKE '%european studies%'
        OR department ILIKE '%jewish studies%'
        OR regexp_matches(department, '\bdesign\b')
        OR department IN ('engl', 'lang')
    THEN 'Humanities & Languages'

    WHEN
        department ILIKE '%anthropology%'
        OR department ILIKE '%social science%'
        OR department ILIKE '%sociology%'
        OR department ILIKE '%political science%'
        OR department ILIKE '%politics%'
        OR department ILIKE '%ethnic stud%'
        OR department ILIKE '%women%'
        OR department ILIKE '%psychology%'
        OR department ILIKE '%psychological%'
        OR department ILIKE '%cognitive science%'
        OR department ILIKE '%global & intl studies%'
        OR department ILIKE '%soc sci undergrad%'
        OR department ILIKE '%comm%'
        OR department ILIKE '%gender%'
        OR department ILIKE '%child%'
        OR department ILIKE '%geography%'
        OR department ILIKE '%human ecology%'
        OR department ILIKE '%social welfare%'
        OR department ILIKE '%global & international studies%'
        OR department ILIKE '%african am studies%'
        OR department ILIKE '%black studies%'
        OR department ILIKE '%social work%'
        OR department ILIKE '%human services%'
        OR department ILIKE '%human development%'
        OR department ILIKE '%criminal justice%'
        OR department ILIKE '%criminology%'
        OR department ILIKE '%international relations%'
        OR department ILIKE '%social & behavioral%'
        OR department ILIKE '%counseling%'
        OR department ILIKE '%conflict resolution%'
        OR department ILIKE '%peacebuilding%'
        OR department ILIKE '%american indian studies%'
        OR department ILIKE '%native amer studies%'
        OR department ILIKE '%race and resistance%'
        OR department ILIKE '%latina/latino studies%'
        OR department ILIKE '%chicana/o studies%'
        OR department ILIKE '%global studies%'
        OR department ILIKE '%int''l studies%'
        OR department ILIKE '%recreation and leisure%'
        OR department ILIKE '%deaf studies%'
        OR department ILIKE '%labor studies%'
        OR department ILIKE '%labor and employment%'
        OR department ILIKE '%queer studies%'
        OR department IN ('soc', 'swrk')
    THEN 'Social Sciences & Communication'

    ELSE 'Other'
END AS dep_bucket
FROM filtered_faculty
"""

duckdb.sql(f"""
CREATE OR REPLACE TABLE faculty_salaries AS
{bucket_sql}
""")

### Save the Cleaned Dataset for EDA
Retain `Other` rows for review. Remove only exact duplicates across the exported columns;
keep separate department affiliations/buckets for the same name and school.
The output is sorted by school, name, year, bucket, and original department.

In [53]:
final_df = duckdb.sql("""
SELECT DISTINCT
    year_transparentCA, school, name, title, role,
    department_scraped AS original_department, dep_bucket AS department,
    totalpay AS total_pay, base AS base_pay,
    overtime AS overtime_pay, other AS other_pay
FROM faculty_salaries
ORDER BY school, name, year_transparentCA, role, department, original_department,
         title, total_pay, base_pay, overtime_pay, other_pay
""").pl()

final_df = final_df.rename({"year_transparentCA": "year", "school": "campus", "total_pay": "gross_pay"})

final_df.write_parquet("../data/processed/golden_csu_2013-2024.parquet")
final_df.describe()

statistic,year,campus,name,title,role,original_department,department,gross_pay,base_pay,overtime_pay,other_pay
str,f64,str,str,str,str,str,str,f64,f64,f64,f64
"""count""",155500.0,"""155500""","""155500""","""146239""","""155500""","""152483""","""155500""",155500.0,155500.0,155500.0,155500.0
"""null_count""",0.0,"""0""","""0""","""9261""","""0""","""3017""","""0""",0.0,0.0,0.0,0.0
"""mean""",2019.607659,null,null,null,null,null,null,57859.55717,52613.403871,0.821293,5245.194682
"""std""",3.224894,null,null,null,null,null,null,52708.87907,47577.989405,86.705861,10159.842805
"""min""",2013.0,"""CSU Bakersfield""","""Aakash Ahamed""","""ASC""","""clinical_adjunct_faculty""","""""","""Agriculture, Food & Animal""",0.0,0.0,0.0,-16941.0
"""25%""",2017.0,null,null,null,null,null,null,0.0,0.0,0.0,0.0
"""50%""",2020.0,null,null,null,null,null,null,56866.0,51802.0,0.0,0.0
"""75%""",2022.0,null,null,null,null,null,null,99498.0,92300.0,0.0,6492.0
"""max""",2024.0,"""Sonoma State University""","""jody berger""","""World Languages Faculty Adviso…","""trainee""","""the School of Journalism and M…","""Social Sciences & Communicatio…",373144.0,373144.0,16722.0,205512.0
